In [1]:
from typing import Any
import librosa
from utils_shiprocket import prepare_data
from utils_shiprocket import give_item_y_and_sr
from utils_shiprocket import evaluate_binary_classifier

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

In [2]:
import numpy as np
from tqdm import tqdm
from typing import Any

class FeatureStore:
    def __init__(
        self,
        audios : list[Any],audio_ids : list[str], sample_rate: int = 22050, hop_length: int = 512,
        n_fft: int = 2048, n_mels: int = 128, n_mfcc: int = 13,
        features: tuple = (
            "log-mel-spectrogram", "mfcc", 'zcr',
            "mfcc-delta", "mfcc-delta2", "pyin","yin")
        ):

        self.n_mels = n_mels
        self.n_mfcc = n_mfcc
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.sr = sample_rate

        if len(audios) == 0:
            raise AttributeError('No audio files provided... , provide at least one..')
        if len(audios) != len(audio_ids):
            raise AttributeError('Number of audio files dont match their ids provided.')

        self.audios = audios
        self.audio_ids = audio_ids

        self.features = features
        self.feature_store = []

    def extract_all(self , verbose = True) -> list[Any]:
        self.feature_store = []
        for a, audio_id in tqdm(zip(self.audios,self.audio_ids), desc = "Extracting Acoustic features", total = len(self.audio_ids), disable = not verbose):
            audio , _ = give_item_y_and_sr(a)
            aux_features_dict = dict()
            # aux_features_dict["audio_id"] = audio_id    # can enable but commented for easier implementation
            # print('Checking and finding MFCC ..')
            need_mfcc = any(
                f in self.features
                for f in ("mfcc", "mfcc-delta", "mfcc-delta2")
            )
            base_mfcc = self._get_mfcc(audio) if need_mfcc else None

            for feature in self.features:
                if feature == "log-mel-spectrogram":
                    # print('Checking and finding log-mel-spectogram..')
                    mel_spec = librosa.feature.melspectrogram(
                        y=audio,
                        sr=self.sr,
                        n_fft=self.n_fft,
                        hop_length=self.hop_length,
                        n_mels=self.n_mels,
                    )
                    aux_features_dict["log-mel-spectrogram"] = librosa.power_to_db(
                        mel_spec, ref=np.max
                    )

                elif feature == "mfcc":
                    # print('Checking and finding mfcc..')
                    aux_features_dict["mfcc"] = base_mfcc

                elif feature == "mfcc-delta":
                    # print('Checking and finding mfcc delta..')
                    aux_features_dict["mfcc-delta"] = librosa.feature.delta(base_mfcc)

                elif feature == "mfcc-delta2":
                    # print('Checking and finding mfcc-delta-delta..')
                    aux_features_dict["mfcc-delta2"] = librosa.feature.delta(
                        base_mfcc, order=2
                    )

                elif feature == "pyin":
                    # print('Checking and finding pyin..')
                    f0, voiced_flag, voiced_probs = librosa.pyin(
                        y= audio,
                        fmin=librosa.note_to_hz("C1"),
                        fmax=librosa.note_to_hz("C7"),
                        sr=self.sr,
                        frame_length=self.n_fft,
                        hop_length=self.hop_length,
                    )
                    aux_features_dict["f0"] = f0.reshape(1, -1)
                    aux_features_dict["voiced_flag"] = voiced_flag.reshape(1, -1)
                    aux_features_dict["voiced_probs"] = voiced_probs.reshape(1, -1)

                elif feature == "yin":
                    # print('Checking and finding yin..')
                    f0 = librosa.yin(
                        y=audio,
                        fmin=librosa.note_to_hz("C1"),
                        fmax=librosa.note_to_hz("C7"),
                        sr=self.sr,
                        frame_length=self.n_fft,
                        hop_length=self.hop_length,
                    )
                    f0= np.nan_to_num(f0, nan=0.0)
                    aux_features_dict["yin_f0"] = f0.reshape(1, -1)

                elif feature == "zcr":
                    # print('Checking and finding zcr..')
                    aux_features_dict["zcr"] = librosa.feature.zero_crossing_rate(
                        y=audio,
                        frame_length=self.n_fft,
                        hop_length=self.hop_length,
                    )
            self.feature_store.append(aux_features_dict)

        return self.feature_store

    def extract_one(self, item) -> dict:
        audio, _ = give_item_y_and_sr(item)
        aux_features_dict = {}

        need_mfcc = any(f in self.features for f in ("mfcc", "mfcc-delta", "mfcc-delta2"))
        base_mfcc = self._get_mfcc(audio) if need_mfcc else None

        for feature in self.features:
            if feature == "log-mel-spectrogram":
                mel_spec = librosa.feature.melspectrogram(y=audio, sr=self.sr, n_fft=self.n_fft, hop_length=self.hop_length, n_mels=self.n_mels)
                aux_features_dict["log-mel-spectrogram"] = librosa.power_to_db(mel_spec, ref=np.max)

            elif feature == "mfcc":
                aux_features_dict["mfcc"] = base_mfcc

            elif feature == "mfcc-delta":
                aux_features_dict["mfcc-delta"] = librosa.feature.delta(base_mfcc)

            elif feature == "mfcc-delta2":
                aux_features_dict["mfcc-delta2"] = librosa.feature.delta(base_mfcc, order=2)

            elif feature == "pyin":
                f0, voiced_flag, voiced_probs = librosa.pyin(y=audio, fmin=librosa.note_to_hz("C1"), fmax=librosa.note_to_hz("C7"), sr=self.sr, frame_length=self.n_fft, hop_length=self.hop_length)

                aux_features_dict["f0"] = np.nan_to_num(f0, nan=0.0).reshape(1, -1)
                aux_features_dict["voiced_flag"] = voiced_flag.reshape(1, -1)
                aux_features_dict["voiced_probs"] = np.nan_to_num(voiced_probs, nan=0.0).reshape(1, -1)

            elif feature == "yin":
                f0 = librosa.yin(y=audio, fmin=librosa.note_to_hz("C1"), fmax=librosa.note_to_hz("C7"), sr=self.sr, frame_length=self.n_fft, hop_length=self.hop_length)
                aux_features_dict["yin_f0"] = np.nan_to_num(f0, nan=0.0).reshape(1, -1)

            elif feature == "zcr":
                aux_features_dict["zcr"] = librosa.feature.zero_crossing_rate(y=audio, frame_length=self.n_fft, hop_length=self.hop_length)

        return aux_features_dict

    def _get_mfcc(self,audio) -> np.ndarray:
        return librosa.feature.mfcc(
            y=audio,
            sr=self.sr,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            n_mfcc=self.n_mfcc,
        )

    def vertical_stack_features_summary_stats(self, verbose = True):
        features_per_item = self.extract_all(verbose)
        summary_stats_features = []
        for features_of_item in tqdm(features_per_item, disable= not verbose):
            stacked_features = np.vstack(list(features_of_item.values()))
            stacked_features = np.nan_to_num(
            stacked_features,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )
            summary_stats_features.append(np.array([
                np.mean(stacked_features, axis=1),
                np.std(stacked_features,axis = 1),
                np.min(stacked_features,axis = 1),
                np.max(stacked_features,axis = 1),
               np.polyfit(np.arange(stacked_features.shape[1]), stacked_features.T, 1)[0], # trend direction
            ]))

        summary_stats_features = np.array(summary_stats_features)
        summary_stats_features = summary_stats_features.reshape(summary_stats_features.shape[0], -1)

        # print(summary_stats_features.shape)
        return summary_stats_features

    def get_temporal_feature(self, item):
        features_of_item = self.extract_one(item)

        if not features_of_item:
            raise ValueError(f"No features extracted. Requested: {self.features}")

        if len(features_of_item) == 1:
            stacked_features = next(iter(features_of_item.values())).T
        else:
            stacked_features = np.vstack(list(features_of_item.values())).T

        return np.nan_to_num(stacked_features, nan=0.0, posinf=0.0, neginf=0.0)

In [3]:
train_df, val_df, test_df = prepare_data(train_examples=None)

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/82 [00:00<?, ?it/s]

In [4]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class AudioDataset(Dataset):
    def __init__(self,df,features_lst):
        self.df = df
        self.features = features_lst

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = self.df[idx]
        fs = FeatureStore([item['audio']],[item['id']], features=self.features)
        X = fs.get_temporal_feature(item)
        Y = item['endpoint_bool']

        return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)

def gru_collate_fn(batch):
    X, y = zip(*batch)

    lengths = torch.tensor([x.shape[0] for x in X], dtype=torch.long) # lenght is neccesary so model ignores it
    X = pad_sequence(X, batch_first=True, padding_value=0.0)
    y = torch.stack(y)

    return X, lengths, y

In [5]:
features =  ("log-mel-spectrogram",) # features are low... deliberately

train_dataset = AudioDataset(train_df,features)
val_dataset = AudioDataset(val_df,features)
test_dataset = AudioDataset(test_df,features)

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=gru_collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=gru_collate_fn,
)

small_dataset = torch.utils.data.Subset(train_dataset, range(16))

small_loader = DataLoader(
    small_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=gru_collate_fn
)

In [7]:
# compute mean/std from REAL frames only
s = torch.zeros(128, dtype=torch.float64)
ss = torch.zeros(128, dtype=torch.float64)
n = 0

for X, lengths, _ in tqdm(train_loader):
    for i, L in enumerate(lengths):
        z = X[i, :L].double()
        s += z.sum(0)
        ss += (z ** 2).sum(0)
        n += L.item()

train_mean = (s / n).float()
train_std = torch.sqrt((ss / n - (s / n) ** 2).clamp(min=1e-6)).float()
train_mean,train_std

100%|██████████| 1946/1946 [10:35<00:00,  3.06it/s]


(tensor([-49.3395, -46.8810, -44.1610, -40.4116, -37.2207, -35.6650, -33.3364,
         -32.6638, -33.7797, -34.3097, -35.2362, -35.2847, -35.1326, -35.3968,
         -34.8736, -35.3132, -35.4490, -35.6433, -36.5462, -36.9102, -38.0512,
         -38.7391, -39.2403, -40.3151, -40.6497, -41.5324, -42.1973, -42.6324,
         -43.5884, -43.7623, -44.3509, -44.9264, -45.0831, -45.7444, -45.7843,
         -46.2208, -46.6753, -46.6265, -47.3254, -47.1641, -47.6662, -47.7810,
         -47.8709, -48.2202, -48.3207, -48.4430, -48.5826, -48.7855, -48.9137,
         -49.2142, -49.3697, -49.2846, -49.3516, -49.3688, -49.4705, -49.6230,
         -49.6621, -49.7279, -49.9616, -50.2239, -50.4318, -50.6236, -50.9279,
         -51.3491, -51.5914, -51.9650, -52.3586, -52.6309, -52.8836, -53.0766,
         -53.2404, -53.4557, -53.5748, -53.7358, -54.0964, -54.4412, -54.7881,
         -55.1254, -55.5296, -55.9668, -56.4659, -56.8685, -57.2837, -57.6785,
         -57.9567, -58.2433, -58.5605, -58.8155, -59

In [10]:
torch.save({"train_mean": train_mean, "train_std": train_std}, "norm_stats.pt")

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

class AudioGRU(nn.Module):
    def __init__(self, input_size, mean, std, hidden_size=64, conv_channels=64):
        super().__init__()
        self.register_buffer("mean", mean)
        self.register_buffer("std", std)
        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, conv_channels, 5, padding=2), nn.ReLU(),
            nn.Conv1d(conv_channels, conv_channels, 3, padding=1), nn.ReLU()
        )
        self.gru = nn.GRU(conv_channels, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths):
        mask = torch.arange(x.size(1), device=x.device)[None, :] < lengths.to(x.device)[:, None]
        x = (x - self.mean) / self.std
        x = x.masked_fill(~mask.unsqueeze(-1), 0.0)

        x = self.cnn(x.transpose(1, 2)).transpose(1, 2)
        x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.gru(x)
        return self.fc(h[-1]).squeeze(1)

X, lengths, y = next(iter(train_loader))

model = AudioGRU(128, train_mean, train_std, hidden_size=64, conv_channels=64)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [14]:
EPOCHS = 5
PRINT_EVERY = 20
val_iter = iter(val_loader)

print(len(train_loader))

try:
    for epoch in range(EPOCHS):
        model.train()

        for batch_idx, (X, lengths, y) in enumerate(train_loader):
            optimizer.zero_grad()
            logits = model(X, lengths)
            train_loss = loss_fn(logits, y)
            train_loss.backward()
            optimizer.step()

            try:
                X_val, val_lengths, y_val = next(val_iter)
            except StopIteration:
                val_iter = iter(val_loader)
                X_val, val_lengths, y_val = next(val_iter)

            model.eval()
            with torch.no_grad():
                val_logits = model(X_val, val_lengths)
                val_loss = loss_fn(val_logits, y_val)
                val_pred = (torch.sigmoid(val_logits) >= 0.5).long()

            y_pred = val_pred.cpu().tolist()
            y_true = y_val.long().cpu().tolist()
            metrics = evaluate_binary_classifier(y_true, y_pred)

            if batch_idx % PRINT_EVERY == 0:
                print(
                    f"Epoch {epoch:02d} | Batch {batch_idx:04d} | "
                    f"train_loss={train_loss.item():.4f} | "
                    f"val_loss={val_loss.item():.4f} | "
                    f"accuracy={metrics['accuracy']:.4f} | "
                    f"f1={metrics['f1_score']:.4f}"
                )

            model.train()

except KeyboardInterrupt:
    print(f"\nStopped at Epoch {epoch}, Batch {batch_idx}")

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "batch_idx": batch_idx
    }, "interrupted_checkpoint.pt")

    print("Checkpoint saved.")

1946
Epoch 00 | Batch 0000 | train_loss=0.4557 | val_loss=0.4880 | accuracy=0.8125 | f1=0.8235


Exception ignored from cffi callback <function SoundFile._init_virtual_io.<locals>.vio_tell at 0x1363d3240>:
Traceback (most recent call last):
  File "/Users/akshat.khatri/PycharmProjects/Shiprocket_final/.venv/lib/python3.11/site-packages/soundfile.py", line 1344, in vio_tell
    @_ffi.callback("sf_vio_tell")

KeyboardInterrupt: 


Epoch 00 | Batch 0020 | train_loss=0.2621 | val_loss=0.4139 | accuracy=0.7500 | f1=0.7778
Epoch 00 | Batch 0040 | train_loss=0.2742 | val_loss=0.1844 | accuracy=0.8750 | f1=0.8750
Epoch 00 | Batch 0060 | train_loss=0.1715 | val_loss=0.2817 | accuracy=0.8750 | f1=0.8750

Stopped at Epoch 0, Batch 68
Checkpoint saved.


In [18]:
checkpoint = torch.load("interrupted_checkpoint.pt", map_location="cpu")
model.load_state_dict(checkpoint["model_state_dict"])
# above only if saved the model

model.eval()

val_loss_total = 0.0
y_true, y_pred, y_prob = [], [], []

dummy_val_dataset = AudioDataset(val_df.take(1000),features)
dummy_loader = DataLoader(
    dummy_val_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=gru_collate_fn
)

with torch.no_grad():
    for X, lengths, y in tqdm(dummy_loader):
        logits = model(X, lengths)
        loss = loss_fn(logits, y)

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()

        val_loss_total += loss.item()
        y_true.extend(y.long().cpu().tolist())
        y_pred.extend(preds.cpu().tolist())
        y_prob.extend(probs.cpu().tolist())

metrics = evaluate_binary_classifier(y_true, y_pred)
avg_val_loss = val_loss_total / len(val_loader)

print(
    f"val_loss={avg_val_loss:.4f} | "
    f"accuracy={metrics['accuracy']:.4f} | "
    f"precision={metrics['precision']:.4f} | "
    f"recall={metrics['recall']:.4f} | "
    f"f1={metrics['f1_score']:.4f}"
)

print(f"true_pos={sum(y_true)}")
print(f"pred_pos={sum(y_pred)}")
print(f"mean_prob={sum(y_prob) / len(y_prob):.4f}")

100%|██████████| 63/63 [00:42<00:00,  1.49it/s]

val_loss=0.0438 | accuracy=0.8410 | precision=0.8048 | recall=0.8688 | f1=0.8356
true_pos=465
pred_pos=502
mean_prob=0.4785
